# 05. Look-Alike Scoring

> **Краткое резюме**: Определяем эталонную группу (лучшие привлечённые клиенты), вычисляем центроид их профиля, и скорим каждого контрагента (проспекта) по похожести на эталон. Результат — ранжированный список проспектов с `lookalike_score`.

**Алгоритм**:
1. **Эталон**: `clientcounterparty_flag = 'Y'` + `clientchange_date IS NOT NULL` (были контрагентами, стали клиентами) + топ по обороту
2. **Центроид**: средний вектор признаков эталонной группы
3. **Scoring**: cosine similarity каждого проспекта к центроиду
4. **Ранжирование**: проспекты (`flag = 'N'`) отсортированы по score

**Предпосылки**: `03_feature_engineering` и `04_segmentation` выполнены.

---

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

OUTPUT_DIR = os.path.abspath('./data')

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

# Доля лучших клиентов для эталона (топ N% по обороту)
TOP_PERCENTILE = 0.2   # 20%

# Минимальный порог score для отнесения к «похожим»
SCORE_THRESHOLD = 0.7

# Количество топ-проспектов для вывода
TOP_N_PROSPECTS = 100

---
## 1. Загрузка данных

In [ ]:
X = pd.read_parquet(os.path.join(OUTPUT_DIR, 'feature_matrix.parquet'))
full_df = pd.read_parquet(os.path.join(OUTPUT_DIR, 'segments.parquet'))

with open(os.path.join(OUTPUT_DIR, 'feature_meta.pkl'), 'rb') as f:
    meta = pickle.load(f)

print(f'Feature matrix: {X.shape}')
print(f'Full data: {full_df.shape}')
print(f'\nDistribution of clientcounterparty_flag:')
print(full_df['clientcounterparty_flag'].value_counts())

---
## 2. Определение эталонной группы

**Лучшие привлечённые клиенты** = те, кто:
1. Сейчас клиент (`clientcounterparty_flag = 'Y'`)
2. Когда-то был контрагентом (`clientchange_date IS NOT NULL`)
3. Имеет высокий оборот (топ N% среди привлечённых)

Если `clientchange_date` не заполнен — используем всех клиентов с высоким оборотом.

In [ ]:
# Клиенты
clients_mask = full_df['clientcounterparty_flag'] == 'Y'
print(f'Clients (Y): {clients_mask.sum():,}')

# Привлечённые (были контрагентами → стали клиентами)
if 'clientchange_date' in full_df.columns:
    acquired_mask = clients_mask & full_df['clientchange_date'].notna()
    print(f'Acquired (Y + clientchange_date): {acquired_mask.sum():,}')
else:
    acquired_mask = clients_mask
    print('clientchange_date not available — using all clients')

# Если мало привлечённых, используем всех клиентов
if acquired_mask.sum() < 100:
    print(f'Too few acquired ({acquired_mask.sum()}), falling back to all clients')
    acquired_mask = clients_mask

# Топ по обороту среди привлечённых
acquired_df = full_df[acquired_mask].copy()
turnover_threshold = acquired_df['total_amount'].quantile(1 - TOP_PERCENTILE)
reference_mask = acquired_mask & (full_df['total_amount'] >= turnover_threshold)

print(f'\nReference group (top {TOP_PERCENTILE*100:.0f}% by turnover): {reference_mask.sum():,}')
print(f'Turnover threshold: {turnover_threshold:,.0f}')

---
## 3. Вычисление центроида и scoring

In [ ]:
# Центроид эталонной группы
X_reference = X.loc[X.index.isin(full_df[reference_mask].index)]
centroid = X_reference.mean(axis=0).values.reshape(1, -1)

print(f'Reference vectors: {X_reference.shape[0]}')

# Cosine similarity ко всем
similarities = cosine_similarity(X.values, centroid).flatten()
full_df['lookalike_score'] = similarities

print(f'\nScore distribution (all):')
print(full_df['lookalike_score'].describe())

---
## 4. Ранжирование проспектов

In [ ]:
# Проспекты = контрагенты (flag = 'N') + неизвестные ('?')
prospects_mask = full_df['clientcounterparty_flag'].isin(['N', '?'])
prospects = full_df[prospects_mask].sort_values('lookalike_score', ascending=False)

print(f'Total prospects: {len(prospects):,}')
print(f'Prospects with score >= {SCORE_THRESHOLD}: {(prospects["lookalike_score"] >= SCORE_THRESHOLD).sum():,}')

# Топ-N
display_cols = [
    'client_name', 'client_type_name', 'clientcounterparty_flag',
    'sparkokved_ccode', 'addrref_city_name', 'srvpackagesegment_ccode',
    'behavioral_segment', 'total_amount', 'unique_counterparties',
    'direction_ratio', 'lookalike_score',
]
display_cols = [c for c in display_cols if c in prospects.columns]

print(f'\nTop-{TOP_N_PROSPECTS} prospects by lookalike_score:')
display(prospects[display_cols].head(TOP_N_PROSPECTS))

In [ ]:
# Распределение scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Все клиенты
for label, mask in [('Clients (Y)', clients_mask), ('Prospects (N/?)', prospects_mask)]:
    subset = full_df.loc[mask, 'lookalike_score'].dropna()
    axes[0].hist(subset, bins=50, alpha=0.5, label=f'{label} ({len(subset):,})', edgecolor='black')
axes[0].axvline(SCORE_THRESHOLD, color='red', linestyle='--', label=f'Threshold {SCORE_THRESHOLD}')
axes[0].set_xlabel('Lookalike Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Score Distribution: Clients vs Prospects')
axes[0].legend()

# Топ-проспекты по сегментам
top_prospects = prospects.head(TOP_N_PROSPECTS)
if 'behavioral_segment' in top_prospects.columns:
    seg_counts = top_prospects['behavioral_segment'].value_counts().sort_index()
    axes[1].bar(seg_counts.index.astype(str), seg_counts.values, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Behavioral Segment')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Top-{TOP_N_PROSPECTS} Prospects by Segment')

plt.tight_layout()
plt.show()

---
## 5. Анализ: профиль top-проспектов vs эталон

In [ ]:
compare_cols = [
    'total_amount', 'tx_count', 'unique_counterparties',
    'direction_ratio', 'active_months',
]
for c in ['avg_balance', 'product_count', 'amount_growth']:
    if c in full_df.columns:
        compare_cols.append(c)

compare = pd.DataFrame({
    'Reference (best clients)': full_df.loc[reference_mask, compare_cols].mean(),
    f'Top-{TOP_N_PROSPECTS} prospects': top_prospects[compare_cols].mean(),
    'All prospects': prospects[compare_cols].mean(),
})
print('Profile comparison:')
display(compare.round(2))

---
## 6. Lift-анализ

Насколько модель лучше случайного выбора?

In [ ]:
# Lift: среди реальных привлечённых (clientchange_date NOT NULL),
# какая доля попадает в топ-K% по score?
if 'clientchange_date' in full_df.columns and acquired_mask.sum() > 0:
    # Ранжируем ВСЕх клиентов по score
    ranked = full_df.sort_values('lookalike_score', ascending=False).copy()
    ranked['rank_pct'] = np.arange(1, len(ranked) + 1) / len(ranked)
    ranked['is_acquired'] = ranked.index.isin(full_df[acquired_mask].index)

    lifts = []
    for pct in [0.05, 0.10, 0.20, 0.30, 0.50]:
        top_k = ranked[ranked['rank_pct'] <= pct]
        acquired_in_top = top_k['is_acquired'].sum()
        expected = acquired_mask.sum() * pct
        lift = acquired_in_top / max(expected, 1)
        lifts.append({'top_%': f'{pct*100:.0f}%', 'acquired_found': acquired_in_top,
                      'expected_random': f'{expected:.0f}', 'lift': f'{lift:.2f}x'})

    print('Lift analysis (acquired clients in top-K% by score):')
    display(pd.DataFrame(lifts))
else:
    print('clientchange_date not available — lift analysis skipped.')

---
## 7. Сохранение результатов

In [ ]:
# Полный результат
scores_path = os.path.join(OUTPUT_DIR, 'lookalike_scores.parquet')
full_df.to_parquet(scores_path)
print(f'Full scores saved: {scores_path}')

# Топ-проспекты отдельно (для менеджеров)
export_cols = [c for c in [
    'client_name', 'client_type_name', 'clientcounterparty_flag',
    'sparkokved_ccode', 'addrref_city_name', 'reg_city_name',
    'srvpackagesegment_ccode', 'behavioral_segment',
    'total_amount', 'tx_count', 'unique_counterparties',
    'direction_ratio', 'active_months', 'amount_growth',
    'lookalike_score',
] if c in prospects.columns]

top_path = os.path.join(OUTPUT_DIR, 'top_prospects.parquet')
prospects[export_cols].head(TOP_N_PROSPECTS * 5).to_parquet(top_path)
print(f'Top prospects saved: {top_path}')

# Excel для менеджеров (если openpyxl доступен)
try:
    xlsx_path = os.path.join(OUTPUT_DIR, 'top_prospects.xlsx')
    prospects[export_cols].head(TOP_N_PROSPECTS * 5).to_excel(xlsx_path)
    print(f'Excel saved: {xlsx_path}')
except Exception as e:
    print(f'Excel export failed ({e}). Install openpyxl if needed.')

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **Эталонная группа** | Лучшие привлечённые клиенты — были контрагентами, стали клиентами, имеют высокий оборот |
| **Центроид** | Средний вектор признаков эталонной группы |
| **Cosine similarity** | Мера похожести векторов [0, 1]: 1 = идентичные, 0 = ортогональные |
| **lookalike_score** | Cosine similarity проспекта к центроиду эталона. Выше = больше похож |
| **Lift** | Во сколько раз модель лучше случайного выбора при отборе топ-K% |
| **clientchange_date** | Дата перехода из контрагентов в клиенты — маркер «привлечённого» |

---

**Pipeline завершён.** Результаты в `data/`.